In [1]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from dotenv import load_dotenv
import os
load_dotenv(override=True)
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash-0731",
)

HTTPS_PROXY: http://127.0.0.1:7897


In [2]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

def get_weather(city:str)->str:
    """get weather for a given city"""
    return f"it's always sunny in {city}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
    checkpointer = InMemorySaver()
)
config = {"configurable":{"thread_id":str(uuid7())}}
stream = agent.stream_events(
    {"messages":[{"role":"user","content":"what is the weather in SF"}]},
    config = config,
    version = "v3",
)
for kind,item in stream.interleave("messages","tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token,end="",flush=True)
    elif kind == "tool_calls":
        print(f"tool call:{item.tool_name} ({item.input})")
        for delta in item.output_deltas:
            print(delta,end="",flush=True)
        print(f"\n Tool_result:{item.output}")

final_state = stream.output


e:\code2\d-api\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
e:\code2\d-api\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


tool call:get_weather ({'city': 'San Francisco'})

 Tool_result:content="it's always sunny in San Francisco" name='get_weather' id='5e7e3bdf-1011-4c63-8aa7-7d2253d606e3' tool_call_id='call_bfe023e5abe740ce941e79a7'
The weather in San Francisco is sunny! ☀️

In [6]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city:str)->str:
    """get weather for a given city"""
    return f"it's always sunny in {city}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "what is the weather in SF"}]},
    stream_mode="messages",
):
    token, metadata = chunk 
    print(f"node: {metadata['langgraph_node']}")
    print(f"role: {token}")


node: model
role: content='' additional_kwargs={'reasoning_content': 'The', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The'}]} response_metadata={'model_provider': 'openrouter'} id='lc_run--01a03cae-3e1d-79f0-85a0-415e7eb8da13' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
node: model
role: content='' additional_kwargs={'reasoning_content': ' user is', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': ' user is'}]} response_metadata={'model_provider': 'openrouter'} id='lc_run--01a03cae-3e1d-79f0-85a0-415e7eb8da13' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
node: model
role: content='' additional_kwargs={'reasoning_content': ' asking about the weather in SF', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': ' asking about the weather in SF'}]} response_metadata={'model_provider': 'openrouter'} id='lc_run--01a03cae-3e1d-79f0-85a0-415e7eb8

In [9]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer

def get_weather(city:str)->str:
    """get weather for a given city"""
    writer = get_stream_writer()
    writer(f"Looking up data to city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"it's always sunny in {city}"
agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools = [get_weather],
)
for chunk in agent.stream(
    {"messages":[{"role":"user","content":"what is the weather in SF"}]},
    stream_mode = "custom",
    version="v3",
):
    print(chunk)

Looking up data to city: San Francisco
Acquired data for city: San Francisco


In [ ]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer

def get_weather(city:str)->str:
    """get weather for a given city"""
    writer = get_stream_writer()
    writer(f"Looking up data to city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"it's always sunny in {city}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools = [get_weather],
)

for chunk in agent.stream(
    {"messages":[{"role":"user","content":"what is the weather in SF"}]},
    stream_mode = ["updates","custom"],
    version="v3",
):
    print(chunk)

('updates', {'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks for weather in SF. I should call get_weather with city "SF" or "San Francisco". Let me use SF.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asks for weather in SF. I should call get_weather with city "SF" or "San Francisco". Let me use SF.'}]}, response_metadata={'model_name': 'deepseek/deepseek-v4-flash-0731', 'id': 'gen-1787725694-mGHtJBoVQjzPQt0xa395', 'created': 1787725694, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.3872e-05, 'cost_details': {'upstream_inference_completions_cost': 9.12e-06, 'upstream_inference_prompt_cost': 4.752e-06, 'upstream_inference_cost': 1.3872e-05}}, id='lc_run--01a03cc1-451b-7ad3-914a-737eca5fa2e9-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_46f7d250cbe84ffc8fbdbb39', 'typ

In [14]:
from langchain.agents import create_agent
from langchain_core.runnables import Runnable


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent: Runnable = create_agent(
    model="openrouter:deepseek/deepseek-r1",
    tools=[get_weather],
)

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    version="v3",
)
for message in stream.messages:
    for token in message.reasoning:
        print(f"[thinking] {token}", end="")
    for token in message.text:
        print(token, end="", flush=True)

[thinking] We[thinking]  are[thinking]  going[thinking]  to[thinking]  call[thinking]  the[thinking]  get[thinking] _[thinking] weather[thinking]  function[thinking]  with[thinking]  the[thinking]  argument[thinking]  '[thinking] San[thinking]  Francisco[thinking] '
[thinking]  However[thinking] ,[thinking]  note[thinking]  that[thinking]  the[thinking]  user[thinking]  said[thinking]  "[thinking] SF[thinking] ",[thinking]  which[thinking]  is[thinking]  an[thinking]  abbreviation[thinking]  for[thinking]  San[thinking]  Francisco[thinking] .
[thinking]  We[thinking]  should[thinking]  convert[thinking]  the[thinking]  city[thinking]  name[thinking]  accordingly[thinking] .
[thinking]  But[thinking]  the[thinking]  function[thinking]  expects[thinking]  a[thinking]  city[thinking]  name[thinking] ,[thinking]  so[thinking]  let[thinking] 's[thinking]  pass[thinking]  '[thinking] San[thinking]  Francisco[thinking] '
I'll check the weather in San Francisco for you.[thinking] 
[thinking] W

In [18]:
from typing import Any
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, AIMessageChunk, AnyMessage, ToolMessage
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """get weather for a given city"""
    return f"it's always sunny in {city}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
)

def _render_chunk(token: AIMessageChunk) -> None:
    if isinstance(token, AIMessageChunk):
        if token.tool_call_chunks:
            print(f"Tool call chunk: {token.tool_call_chunks}")
        elif token.content:
            print(token.content, end="", flush=True)

def _render_update(update: dict) -> None:
    for node, state in update.items():
        for msg in state.get("messages", []):
            if isinstance(msg, ToolMessage):
                print(f"\nTool result [{msg.name}]: {msg.content}")

input_message = HumanMessage(content="what is the weather in SF")
for chunk in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
):
    mode, token = chunk
    if mode == "messages":
        token, metadata = token
        _render_chunk(token)
    elif mode == "updates":
        _render_update(token)

Tool call chunk: [{'name': 'get_weather', 'args': '', 'id': 'call_a73add10529140c3903cdcb1', 'index': 0, 'type': 'tool_call_chunk'}]
Tool call chunk: [{'name': None, 'args': '', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool call chunk: [{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool call chunk: [{'name': None, 'args': '"city": "San', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool call chunk: [{'name': None, 'args': ' Francisco"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]

Tool result [get_weather]: it's always sunny in San Francisco
The weather in San Francisco is sunny — it's always sunny there! ☀️

In [20]:
from typing import Any,Literal
from langchain.agents.middleware import after_agent ,AgentState
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from pydantic import BaseModel

class ResponseSafety(BaseModel):
    """Evaluate a response as sage or unsafe"""
    evaluation: Literal["safe","unsafe"]

safety_model = init_chat_model(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
)

@after_agent(can_jump_to=["end"])
def safety_guardrail(state:AgentState,runtime:Runtime)->dict[str,Any]:
    """Model-based guardrail : use an LLM to evaluate response safety"""
    stream_writer = get_stream_writer()
    if not state["messages"]:
        return None
    last_message = state["messages"][-1]
    if not isinstance(last_message,AIMessage):
        return None

    model_with_tools = safety_model.bind_tools([ResponseSafety],tool_choice="any")
    result = model_with_tools.invoke(
        [
            {
                "role": "system",
                "content": "Evaluate this AI response as generally safe or unsafe."
            },
            {
                "role": "user",
                "content": f"AI response: {last_message.text}"
            }
        ]
    )
    stream_writer(result)
    tool_call = result.tool_calls[0]
    if tool_call["args"]["evaluation"] == "unsafe":
        last_message.content = "I cannot provide that response. Please rephrase your request."

    return None

In [27]:
from typing import Any
from langchain.agents import create_agent
from langchain.messages import AIMessageChunk, AIMessage, AnyMessage, ToolMessage
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """get weather for a given city"""
    return f"it is always sunny in {city}"

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
    middleware=[safety_guardrail],
)

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.content:
        print(token.content, end="", flush=True)
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content}")

input_message = {"role": "user", "content": "What is the weather in Boston?"}
for chunk in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates", "custom"],
):
    print(chunk)
    mode, data = chunk
    if mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    elif mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])
    elif mode == "custom":
        print(f"Safety check result: {data}")

('messages', (AIMessageChunk(content='', additional_kwargs={'reasoning_content': 'The', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The'}]}, response_metadata={'model_provider': 'openrouter'}, id='lc_run--01a03d07-a059-7d12-9670-a47cae0f00d1', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'ls_integration': 'langchain_chat_model', 'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:5a650392-e900-43ec-96d3-34ddfe10da46', 'checkpoint_ns': 'model:5a650392-e900-43ec-96d3-34ddfe10da46', 'ls_provider': 'openrouter', 'ls_model_name': 'deepseek/deepseek-v4-flash-0731', 'ls_model_type': 'chat', 'ls_temperature': None, 'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-openrouter': '0.2.7'}}))
('messages', (AIMessageChunk(content='', additional_kwargs={'reasoning_content': ' user asks for wea

In [29]:
input_message = {"role":"user","content":"what is the weather in Boston"}
full_message = None
for chunk in agent.stream(
    {"messages":[input_message]},
    stream_mode = ["messages","updates"],
):
    mode, data = chunk
    if mode == "messages":
        token, metadata = data
        if isinstance(token,AIMessageChunk):
            _render_message_chunk(token)
            full_message = token if full_message is None else full_message + token
            if token.chunk_position == "last":
                if full_message.tool_calls:
                    print(f"Tool calls :{full_message.tool_calls}")
                full_message = None
    elif mode == "updates":
        for source, update in data.items():
            if source == "tools":
                _render_completed_message(update["messages"][-1])


[{'name': 'get_weather', 'args': '', 'id': 'call_c52331930b0f4d829b6cc5f0', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"city": "Boston"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool calls :[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_c52331930b0f4d829b6cc5f0', 'type': 'tool_call'}]
Tool response: it is always sunny in Boston
The weather in Boston is always sunny! ☀️[{'name': 'ResponseSafety', 'args': '', 'id': 'call_c4971d0791d243ceb593ab61', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"evaluation": "safe"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool calls :[{'name': 'ResponseSaf

In [ ]:
# 初见HumanInTheLoopMiddleware 原理是在见到指定标志的时候暂停工作流 等下一个输入之后进行 工作流的重连（再调用）
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import AIMessageChunk, ToolMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.utils.uuid import uuid7

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to someone."""
    return f"Email sent to {to}: [{subject}] {body}"

checkpointer = InMemorySaver()

agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"send_email": True}),
    ],
    checkpointer=checkpointer,  # 必须有，才能暂停后恢复
)

config = {"configurable": {"thread_id": str(uuid7())}}

print("=== Agent 开始运行 ===")
interrupts = []

for mode, data in agent.stream(
    {"messages": [{"role": "user", "content": "帮我给 boss@company.com 发一封邮件，说明天我要请假"}]},
    config=config,
    stream_mode=["messages", "updates"],
):
    if mode == "messages":
        token, _ = data
        if isinstance(token, AIMessageChunk) and token.content:
            print(token.content, end="", flush=True)
    elif mode == "updates":
        if "__interrupt__" in data:
            for interrupt_obj in data["__interrupt__"]:
                interrupts.append(interrupt_obj)
                print("\n⏸️  Agent 暂停，等待审批：")
                for req in interrupt_obj.value["action_requests"]:
                    print(f"   工具: {req['name']}")
                    print(f"   参数: {req['args']}")

print(f"\n捕获到 {len(interrupts)} 个中断")

=== Agent 开始运行 ===
The user wants me to send an email to boss@company.com saying they will take leave tomorrow. The email content should be in Chinese since the user is writing in Chinese.

Let me compose the email. The user didn't specify details, so I'll keep it simple.

Let me send the email.我来帮您发送这
⏸️  Agent 暂停，等待审批：
   工具: send_email
   参数: {'to': 'boss@company.com', 'subject': '请假申请', 'body': '老板您好：\n\n我明天需要请假一天，特此向您申请，请批准。\n\n谢谢！'}

捕获到 1 个中断


In [37]:
from langgraph.types import Command

decisions = {}
for interrupt_obj in interrupts:
    decisions[interrupt_obj.id] = {
        "decisions": [
            {
                "type": "edit",
                "edited_action": {
                    "name": "send_email",
                    "args": {
                        "to": "boss@company.com",
                        "subject": "[请假申请] 明天请假",
                        "body": "您好，我明天因个人原因需要请假一天，请批准。谢谢！",
                    },
                },
            }
        ]
    }

print("=== 恢复执行 ===")

for mode, data in agent.stream(
    Command(resume=decisions),
    config=config,   # 同一个 thread_id，接上之前的状态
    stream_mode=["messages", "updates"],
):
    if mode == "messages":
        token, _ = data
        if isinstance(token, AIMessageChunk) and token.content:
            print(token.content, end="", flush=True)
    elif mode == "updates":
        for source, update in data.items():
            if source == "tools":
                msg = update["messages"][-1]
                if isinstance(msg, ToolMessage):
                    print(f"\n✅ 工具执行结果: {msg.content}")

print("\n=== 完成 ===")

=== 恢复执行 ===

✅ 工具执行结果: Email sent to boss@company.com: [[请假申请] 明天请假] 您好，我明天因个人原因需要请假一天，请批准。谢谢！
邮件已经发送给您老板 boss@company.com 了。

邮件内容如下：
- **主题**：[请假申请] 明天请假
- **正文**：您好，我明天因个人原因需要请假一天，请批准。谢谢！

如果您需要补充请假原因、调整日期或添加其他内容，随时告诉我，我可以帮您修改后重新发送。
=== 完成 ===


In [50]:
from ast import In
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.messages import AIMessage,AIMessageChunk,HumanMessage,AnyMessage,ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command,Interrupt

@tool
def get_weather(city:str)->str:
    """get weather for a given city"""
    return f"it is always sunny in {city}"

checkpointer = InMemorySaver()
agent = create_agent(
    model="openrouter:deepseek/deepseek-v4-flash-0731",
    tools=[get_weather],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"get_weather": True})],
    checkpointer = checkpointer,
)
def _render_message_chunk(token:AIMessageChunk)->None:
    if token.text:
        print(token.text,end="|",)
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")

def _render_interrupt(interrupt:Interrupt) ->None:
    interrupts = interrupt.value
    for request in interrupts["action_requests"]:
        print(request["description"])

input_message = HumanMessage("Can you look up the weather in Boston and San Francisco?")
config = {"configurable":{"thread_id":"DETS001"}}
interrupts = []
for mode, data in agent.stream(
    {"messages":[input_message]},
    config = config,
    stream_mode=["messages","updates"],
):
    if mode == "messages":
        token,metadata = data
        if isinstance(token,AIMessageChunk):
            _render_message_chunk(token)
    elif mode == "updates":
        for source,update in data.items():
            if source in ("model","tools"):
                _render_completed_message(update["messages"][-1])
            if source == "__interrupt__":
                interrupts.extend(update)
                _render_interrupt(update[0])




I|'ll look| up the weather for both cities| right away|[{'name': 'get_weather', 'args': '', 'id': 'call_60dcbc95236047d59aea60db', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"city": "Boston"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': 'get_weather', 'args': '', 'id': 'call_6d0d91f98c8149ac97de2295', 'index': 1, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '', 'id': None, 'index': 1, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 1, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"city": "San', 'id': None, 'index': 1, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' Francisco"}', 'id': None, 'index': 1, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_60dcbc95236047d59aea60db', 'type'

In [51]:
def _get_interrupt_decision(interrupt:Interrupt)->list[dict]:
    return [
        {
            "type":"edit",
            "edited_action":{
                "name":"get_weather",
                "args":{"city":"Boston,U.K."},
            },
        }
    ]
decisions = {}
for interrupt in interrupts:
    decisions[interrupt.id]={
        "decisions":_get_interrupt_decision(interrupt)
    }

print(decisions)

{'c608219beb5c8fcdcbba6e24ec79123d': {'decisions': [{'type': 'edit', 'edited_action': {'name': 'get_weather', 'args': {'city': 'Boston,U.K.'}}}]}}


In [52]:
from langgraph.types import Command
from langchain.messages import AIMessageChunk, ToolMessage

decisions = {}

for mode, data in agent.stream(
    Command(resume=decisions),
    config=config,
    stream_mode=["messages", "updates"],
):
    if mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            if token.content:
                print(token.content, end="", flush=True)
    elif mode == "updates":
        for source, update in data.items():
            if source == "tools":
                for msg in update["messages"]:
                    if isinstance(msg, ToolMessage):
                        print(f"\n✅ {msg.name} 结果: {msg.content}")

print("\n=== 完成 ===")


=== 完成 ===


In [60]:
from typing import Any
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, AnyMessage
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """get weather for a given city"""
    return f"it is always sunny in {city}"

weather_model = init_chat_model("openrouter:deepseek/deepseek-v4-flash-0731")
weather_agent = create_agent(model=weather_model, tools=[get_weather])  # 要调用！

@tool
def call_weather_agent(query: str) -> str:
    """Query the weather agent"""
    result = weather_agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    return result["messages"][-1].text  # messages 加 s

supervisor_model = init_chat_model("openrouter:deepseek/deepseek-v4-flash-0731")
agent = create_agent(
    model=supervisor_model,
    tools=[call_weather_agent],
    name="supervisor"
)

In [61]:
def _render_message_chunk(token:AIMessageChunk)->None:
    if token.text:
        print(token.text,end="")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

def _render_completed_message(message:AnyMessage)->None:
    if isinstance(message,AIMessage) and message.tool_calls:
        print(f"Tool calls:{message.tool_calls}")
    if isinstance(message,ToolMessage):
        print(f"Tool response:{message.content_blocks}")

input_message = {"role":"user","content":"What is the weather in Boston"}
current_agent = None
for namespace,mode,data in agent.stream(
    {"messages":[input_message]},
    stream_mode=["messages","updates"],
    subgraphs=True,
):
    if mode == "messages":
        token,metadata = data
        if agent_name := metadata.get("lc_agent_name"):
            if agent_name != current_agent:
                print(f"{agent_name}")
                current_agent = agent_name
        if isinstance(token,AIMessageChunk):
            _render_message_chunk(token)
    elif mode == "updates":
        for source,update in data.items():
            if source in ("model","tools"):
                _render_completed_message(update["messages"][-1])



supervisor
[{'name': 'call_weather_agent', 'args': '', 'id': 'call_fe4069b824284b2ba482bdd4', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"query": "weath', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'er in Boston"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
Tool calls:[{'name': 'call_weather_agent', 'args': {'query': 'weather in Boston'}, 'id': 'call_fe4069b824284b2ba482bdd4', 'type': 'tool_call'}]
[{'name': 'get_weather', 'args': '', 'id': 'call_7c78d388d42140f5a6154611', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '"city": "Boston"}', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}